In [8]:
from LanguageDatasets import LanguageDataset
from groq import Groq
from dotenv import load_dotenv
import os
import random
import time
import pandas as pd

### 3. Corrección gramatical mediante introducción de errores

Para evaluar la capacidad de corrección, se parte de textos correctos en asturiano o aranés (por ejemplo, de Wikipedia). Un LLM grande introduce errores controlados de ortografía, morfología o sintaxis. El modelo evaluado debe corregirlos. La comparación con el texto original permite medir la calidad de la corrección.

- `"llama-3.3-70b-versatile"` Mucho más rápido
- `"openai/gpt-oss-120b"` Para producción?

In [9]:
def safe_chat_completion(client, model, messages, sleep_time=0.5, max_retries=5): 
    """ Llama a client.chat.completions.create con reintentos automáticos. Si falla (por ejemplo, error 429 o timeout), espera sleep_time y reintenta. """ 
    for attempt in range(max_retries): 
        try: 
            return client.chat.completions.create( model=model, messages=messages ) 
        except Exception as e: # Último intento → relanzar error 
            if attempt == max_retries - 1: 
                raise e # Espera antes del siguiente intento 
            wait = sleep_time * (attempt + 1) # backoff lineal 
            print(f"Error en intento {attempt+1}: {e}. Reintentando en {wait} segundos...") 
            time.sleep(wait)


In [14]:
def generateDatasetOrtografico(dataset: LanguageDataset, api_key, model="openai/gpt-oss-120b", save = True, max_errors=4, min_errors=0, sleep_time=0.15, max_retries=5):
    client = Groq(api_key=api_key)
    res_list = []
    i = 0
    total = len(dataset)
    for original in dataset:
        callBegin = time.time()
        n_errors = random.randint(min_errors, max_errors)
        modified = safe_chat_completion(client, sleep_time=sleep_time, max_retries=max_retries,
            model=model,
            messages=[
                {"role": "user", "content": f"Modifica esta frase, añadiendole {n_errors} errores gramaticales, ortográficos o léxicos. No añadas más contenido a la frase ni cambies el significado. Devuelve solo la frase modificada: '{original["text"]}'"}
            ]
        )
        res_list.append((original["text"], modified.choices[0].message.content, n_errors))
        callEnd = time.time()
        # ---- PROGRESO ---- 
        i += 1
        pct = (i / total) * 100 
        step = int(callEnd - callBegin)
        remaining = step * (total - i)
        print(f"\rProgreso: {pct:5.1f}% ({i}/{total}) | step: {step} s, remaining time: {remaining// 60} min y {remaining - 60 * (remaining // 60)} s", end="") 
    print("\rDataset Generado") # salto de línea al terminar

    res_df = pd.DataFrame(res_list, columns=["original","modified", "n_errors"])
    if save:
        date = time.localtime(time.time())
        res_df.to_csv(f"{dataset.language}_{model.split("/")[-1]}_{time.strftime("%m-%d_%H-%M-%S", date)}")
    return res_df

In [ ]:
load_dotenv("secrets.env")
ast = LanguageDataset("asturiano",True)
evalDataset = generateDatasetOrtografico(ast, os.getenv("GROQ_API_KEY"), save=False, model="llama-3.3-70b-versatile")

Descargando tatoeba para asturiano:
Completado con éxito
Progreso:   2.2% (10/448) | step: 2 s, remaining time: 14 min y 36 s

In [ ]:
evalDataset.shape

(448, 3)

In [ ]:
evalDataset.head()

,original,modified,1
0,Tas buenu pa dir a nengún sitiu.,"'Tas buenu pa dir a nengun sitiou, porque toi ...",4
1,El cielu del atapecer ye roxu.,"El cielu del atardecer es roxo, el barco navev...",3
2,Nun gastes más perres de les que ganes.,'Nun gastis más perros de les que ganas.',4
3,Lleva-y les llaves al to hermanu.,Lleva-y les llabe al tu hermano.,2
4,Esti xergón ye vieyu y máncame nel renaz.,Esti xergon ye viejyo y mancame en el renaz y ...,2


In [ ]:
evalDataset.n_errors.value_counts()